# Phase 4: Feature Engineering & Preprocessing

This notebook loads the raw merged data, implements local median gap filling and large gap dropping, extracts Continuous Wavelet Transform (CWT) features using PyWavelets with the Ricker (`mexh`) wavelet, plots a CWT scalogram (replicating Figure 1b), and exports the preprocessed original and wavelet feature sets.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt

sys.path.append(os.path.abspath('../'))
from src.data_loader import load_las_dataset
from src.features import handle_missing_values, compute_cwt_features

# 1. Load merged dataset
df = load_las_dataset('../data/raw')
print(f'Raw dataset shape: {df.shape}')

## 1. Missing Value Imputation

Small gaps (< 5 consecutive NaN samples) are imputed with each well's median value of that log. Large gaps or logs missing entirely are dropped.

In [ ]:
feature_cols = ['DEPTH_MD', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'BS']
df_clean = handle_missing_values(df, feature_cols + ['LITHOLOGY'])
print(f'Cleaned dataset shape: {df_clean.shape}')

## 2. Continuous Wavelet Transform (CWT) Feature Extraction

Decomposes geological signal into local frequency components. We apply it to the 7 target logs per well.

In [ ]:
target_logs = ['GR', 'NPHI', 'SP', 'RDEP', 'RHOB', 'DTC', 'PEF']
df_features = compute_cwt_features(df_clean, target_logs)
print(f'Dataset with CWT shape: {df_features.shape}')
df_features.head()

## 3. CWT Scalogram Visualization

Plots the full 2D continuous wavelet decomposition coefficients over different scales for a selected well section (replicating Figure 1b from the paper).

In [ ]:
well_name = df_features['WELL_ID'].unique()[0]
well_df = df_features[df_features['WELL_ID'] == well_name].sort_values('DEPTH_MD')
signal = well_df['GR'].values[:500]
scales = np.arange(1, 31)

coefs, freqs = pywt.cwt(signal, scales, 'mexh')

plt.figure(figsize=(10, 6))
plt.imshow(np.abs(coefs), extent=[0, len(signal), 1, 30], cmap='PRGn', aspect='auto',
           vmax=abs(coefs).max(), vmin=-abs(coefs).max())
plt.colorbar(label='CWT Coefficient Magnitude')
plt.xlabel('Sample Index (0.1 m steps)')
plt.ylabel('Wavelet Scale')
plt.title(f'Continuous Wavelet Transform (CWT) Scalogram of GR (Well: {well_name})', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/gr_cwt_scalogram.png', dpi=150)
plt.show()

## 4. Exporting Interim Datasets

Saves the engineered features as Parquet for training.

In [ ]:
os.makedirs('../data/interim', exist_ok=True)
df_features.to_parquet('../data/interim/processed_features.parquet', index=False)
print('Engineered features successfully saved to data/interim/processed_features.parquet!')